In [ ]:
# =============================================
# 1. استيراد المكتبات
# =============================================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive

# =============================================
# 2. إعداد الجهاز (CPU أو GPU)
# =============================================
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU found. Using CPU instead.")

print(f"🖥️ Using device: {device}")

# =============================================
# 3. التحويلات وزيادة البيانات (Data Augmentation)
# =============================================
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

✅ GPU detected: Tesla T4
🖥️ Using device: cuda


In [ ]:
drive.mount('/content/drive')

# تحميل البيانات
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = datasets.ImageFolder(root="/content/drive/MyDrive/database/", transform=transform)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# =============================================
# 5. نموذج CNN + LSTM (محدث)
# =============================================
class CNN_LSTM_Model(nn.Module):
    def __init__(self, num_classes=2):
        super(CNN_LSTM_Model, self).__init__()

        # CNN Layers
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # LSTM Layer
        self.lstm = nn.LSTM(
            input_size=128 * 8,  # بعد Flatten للعرض
            hidden_size=128,
            num_layers=1,
            batch_first=True
        )

        # Fully Connected Layers
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)             # (B, 128, 8, 8)
        x = x.view(x.size(0), 8, -1)  # (B, 8, 1024)
        lstm_out, _ = self.lstm(x)  # (B, 8, 128)
        out = self.fc(lstm_out[:, -1, :])
        return out

In [ ]:

# =============================================
# 6. التهيئة
# =============================================
model = CNN_LSTM_Model(num_classes=len(dataset.classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
patience = 5

train_losses, val_losses, train_accs, val_accs = [], [], [], []
precisions, recalls, f1s = [], [], []

best_f1 = 0
best_epoch = 0
no_improve_count = 0

In [ ]:

# =============================================
# 7. التدريب مع Early Stopping
# =============================================
for epoch in range(epochs):
    # -------- تدريب --------
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    train_loss = running_loss / total
    train_acc = correct / total

    # -------- تحقق --------
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    val_loss /= val_total
    val_acc = val_correct / val_total

    # -------- Precision / Recall / F1 --------
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"P: {precision:.4f} | R: {recall:.4f} | F1: {f1:.4f}")

    # -------- Early Stopping --------
    if f1 > best_f1:
        best_f1 = f1
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "best_cnn_lstm_model.pth")
        no_improve_count = 0
    else:
        no_improve_count += 1

    if no_improve_count >= patience:
        print(f"\n⏹️  Early stopping at epoch {epoch+1}")
        break

print(f"\n✅ أفضل F1-Score: {best_f1:.4f} عند epoch {best_epoch}")

In [ ]:

# =============================================
# 8. تقييم على مجموعة الاختبار + مصفوفة الالتباس
# =============================================
model.load_state_dict(torch.load("best_cnn_lstm_model.pth"))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# حساب المقاييس النهائية
precision_test = precision_score(all_labels, all_preds, average='macro')
recall_test = recall_score(all_labels, all_preds, average='macro')
f1_test = f1_score(all_labels, all_preds, average='macro')

print(f"\nTest Precision: {precision_test:.4f}")
print(f"Test Recall:    {recall_test:.4f}")
print(f"Test F1-Score:  {f1_test:.4f}")

# ====== مصفوفة الالتباس ======
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=dataset.classes)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title("Confusion Matrix")
plt.savefig("confusion_matrix.png")
plt.show()
print("✅ تم حفظ مصفوفة الالتباس في confusion_matrix.png")

# =============================================
# 9. حفظ النتائج في CSV
# =============================================
metrics_df = pd.DataFrame({
    "Epoch": list(range(1, len(train_losses)+1)),
    "Train_Loss": train_losses,
    "Val_Loss": val_losses,
    "Train_Acc": train_accs,
    "Val_Acc": val_accs,
    "Precision": precisions,
    "Recall": recalls,
    "F1_Score": f1s
})
metrics_df.to_csv("metrics_results.csv", index=False)
print("✅ تم حفظ جميع المقاييس لكل epoch في metrics_results.csv")